# **Mapping Residential Development Trajectories as Indicators of Emerging and Future Business Markets**

Week 2

Data Source: https://www.census.gov/construction/bps/msamonthly.html

•	Data Access – Downloading data on the CBSA (area with either population in the range 10,000-50,000) or ≥50,000) for each month since 2020 totaling to 79 months for 921 CBSAs.

•	Data Cleaning – Creating the function for looping over the downloaded data files and cleaning each of the csv files.

•	Data Aggregation – Combining the clean 79 .xls files into a single merged file for both the units and value datasets.


In [1]:
%pip install pandas xlrd requests beautifulsoup4 pyarrow

## **Data Links Extraction from BPS and Storage Initialization**

In [2]:
from pathlib import Path
from urllib.parse import urljoin
import io
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup


INDEX_URL = "https://www.census.gov/construction/bps/msamonthly.html"

START_YEAR = 2020
END_YEAR = 2026

BASE_DIR = Path("bps_data")
RAW_DIR = BASE_DIR / "raw"

UNITS_OUTPUT = BASE_DIR / "bps_msa_units_monthly.parquet"
VALUE_OUTPUT = BASE_DIR / "bps_msa_value_monthly.parquet"

RAW_DIR.mkdir(parents=True, exist_ok=True)


Function for getting monthly data files

In [3]:
def get_monthly_files():

    response = requests.get(INDEX_URL, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    files = []

    for a in soup.find_all("a", href=True):

        href = a["href"]
        text = a.get_text(" ", strip=True)

        # Excel files only
        if not href.lower().endswith(".xls"):
            continue

        # Explicitly excluding Annual files
        if "annual" in text.lower():
            continue

        url = urljoin(INDEX_URL, href)

        # Extracting year/month from the link text or URL
        match = re.search(
            r"(January|February|March|April|May|June|July|August|"
            r"September|October|November|December)\s+(\d{4})",
            text,
            re.I
        )

        if not match:
            continue

        month_name = match.group(1)
        year = int(match.group(2))

        month = pd.to_datetime(
            month_name,
            format="%B"
        ).month

        if START_YEAR <= year <= END_YEAR:

            files.append({
                "year": year,
                "month": month,
                "date": pd.Timestamp(year, month, 1),
                "url": url,
            })

    return (
        pd.DataFrame(files)
        .drop_duplicates(["year", "month"])
        .sort_values(["year", "month"])
        .reset_index(drop=True)
    )


In [4]:
files = get_monthly_files()

print(files)
print(f"\nMonthly files: {len(files)}")

    year  month       date                                                url
0   2020      1 2020-01-01  https://www.census.gov/construction/bps/xls/ms...
1   2020      2 2020-02-01  https://www.census.gov/construction/bps/xls/ms...
2   2020      3 2020-03-01  https://www.census.gov/construction/bps/xls/ms...
3   2020      4 2020-04-01  https://www.census.gov/construction/bps/xls/ms...
4   2020      5 2020-05-01  https://www.census.gov/construction/bps/xls/ms...
..   ...    ...        ...                                                ...
74  2026      3 2026-03-01  https://www.census.gov/construction/bps/xls/cb...
75  2026      4 2026-04-01  https://www.census.gov/construction/bps/xls/cb...
76  2026      5 2026-05-01  https://www.census.gov/construction/bps/xls/cb...
77  2026      6 2026-06-01  https://www.census.gov/construction/bps/xls/cb...
78  2026      7 2026-07-01  https://www.census.gov/construction/bps/xls/cb...

[79 rows x 4 columns]

Monthly files: 79


## **Data Extraction from BPS**

In [6]:
def download_file(row):

    filename = (
        f"{row['year']}_{row['month']:02d}.xls"
    )

    path = RAW_DIR / filename

    if path.exists():
        return path

    print(f"Downloading {filename}")

    response = requests.get(
        row["url"],
        timeout=60
    )
    response.raise_for_status()

    path.write_bytes(response.content)

    return path


Parsing each worksheet

In [13]:
def read_sheet(
    file_path,
    sheet_name,
    year,
    month
):

    # Reading Row 8 as the header
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=7,
        engine="xlrd" # adding engine='xlrd' for .xls files
    )

    # -----------------------------------------------------
    # 2020-2023: A:I
    # -----------------------------------------------------

    if year <= 2023:
        df = df.iloc[:, :9]

    # -----------------------------------------------------
    # 2024+: A:J, then removing D
    # -----------------------------------------------------

    else:
        df = df.iloc[:, :10]

        # Column D = positional index 3
        df = df.drop(
            df.columns[3],
            axis=1
        )

    # Removing completely blank rows
    df = df.dropna(how="all")

    # Cleaning column names
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Adding time dimensions
    df["year"] = year
    df["month"] = month
    df["date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    return df


Extracting both MSA Units and MSA Value datasets

In [16]:
units_data = []
value_data = []

for _, row in files.iterrows():
    year = int(row["year"])
    month = int(row["month"])

    print(f"Processing {year}-{month:02d}")

    try:
        file_path = download_file(row)

        xls = pd.ExcelFile(file_path, engine="xlrd") # Re-adding engine='xlrd' for .xls files
        sheet_names = xls.sheet_names

        units_sheet_name = None
        if "MSA Units" in sheet_names:
            units_sheet_name = "MSA Units"
        elif "CBSA Units" in sheet_names:
            units_sheet_name = "CBSA Units"
        else:
            raise ValueError(f"No valid units sheet found in {file_path}. Available sheets: {sheet_names}")

        value_sheet_name = None
        if "MSA Value" in sheet_names:
            value_sheet_name = "MSA Value"
        elif "CBSA Value" in sheet_names:
            value_sheet_name = "CBSA Value"
        else:
            raise ValueError(f"No valid value sheet found in {file_path}. Available sheets: {sheet_names}")

        # Read Units sheet
        units = read_sheet(
            file_path=file_path,
            sheet_name=units_sheet_name,
            year=year,
            month=month
        )

        # Read Value sheet
        value = read_sheet(
            file_path=file_path,
            sheet_name=value_sheet_name,
            year=year,
            month=month
        )

        units_data.append(units)
        value_data.append(value)

    except Exception as e:
        print(f"Error processing file for {year}-{month:02d} (URL: {row['url']}): {e}")
        # Instead of re-raising, log the error and continue to the next file
        continue


Processing 2020-01
Processing 2020-02
Processing 2020-03
Processing 2020-04
Processing 2020-05
Processing 2020-06
Processing 2020-07
Processing 2020-08
Processing 2020-09
Processing 2020-10
Processing 2020-11
Processing 2020-12
Processing 2021-01
Processing 2021-02
Processing 2021-03
Processing 2021-04
Processing 2021-05
Processing 2021-06
Processing 2021-07
Processing 2021-08
Processing 2021-09
Processing 2021-10
Processing 2021-11
Processing 2021-12
Processing 2022-01
Processing 2022-02
Processing 2022-03
Processing 2022-04
Processing 2022-05
Processing 2022-06
Processing 2022-07
Processing 2022-08
Processing 2022-09
Processing 2022-10
Processing 2022-11
Processing 2022-12
Processing 2023-01
Processing 2023-02
Processing 2023-03
Processing 2023-04
Processing 2023-05
Processing 2023-06
Processing 2023-07
Processing 2023-08
Processing 2023-09
Processing 2023-10
Processing 2023-11
Processing 2023-12
Processing 2024-01
Processing 2024-02
Processing 2024-03
Processing 2024-04
Processing 2

## **Data Aggregation** : concatenate

In [17]:
# Concatenate
msa_units = pd.concat(
    units_data,
    ignore_index=True
)

msa_value = pd.concat(
    value_data,
    ignore_index=True
)


In [18]:
# Checking Data
print("Units:")
print(msa_units.shape)
print(msa_units.columns.tolist())

print("\nValue:")
print(msa_value.shape)
print(msa_value.columns.tolist())


Units:
(46601, 12)
['CSA', 'CBSA', 'Name', 'Total', '1 Unit', '2 Units', '3 and 4 Units', '5 Units or More', 'Num of Structures With 5 Units or More', 'year', 'month', 'date']

Value:
(46601, 13)
['CSA', 'CBSA', 'Name', 'Total', '1 Unit', '2 Units', '3 and 4 Units', '5 Units or More', 'Unnamed: 8', 'year', 'month', 'date', 'Unnamed: 9']


In [19]:
print(
    msa_value
    .groupby(["year", "month"])
    .size()
)


year  month
2020  1        368
      2        368
      3        368
      4        368
      5        368
              ... 
2026  3        921
      4        922
      5        922
      6        921
      7        921
Length: 79, dtype: int64


In [20]:
expected = pd.date_range(
    start=f"{START_YEAR}-01-01",
    end=f"{END_YEAR}-12-01",
    freq="MS"
)

actual = pd.to_datetime(
    msa_units["date"].unique()
)

missing = expected.difference(actual)

print("Missing months:")
print(missing)


Missing months:
DatetimeIndex(['2026-08-01', '2026-09-01', '2026-10-01', '2026-11-01',
               '2026-12-01'],
              dtype='datetime64[ns]', freq='MS')


In [22]:
# Saving Datasets to Parquet
msa_units.to_parquet(
    UNITS_OUTPUT,
    index=False
)

msa_value.to_parquet(
    VALUE_OUTPUT,
    index=False
)

# Defining output paths for CSV files
UNITS_CSV_OUTPUT = BASE_DIR / "bps_msa_units_monthly.csv"
VALUE_CSV_OUTPUT = BASE_DIR / "bps_msa_value_monthly.csv"

# Saving Datasets to CSV
msa_units.to_csv(
    UNITS_CSV_OUTPUT,
    index=False
)

msa_value.to_csv(
    VALUE_CSV_OUTPUT,
    index=False
)


In [23]:
import zipfile
from google.colab import files

# Defining the path to the folder to be zipped
folder_path = BASE_DIR
zip_file_name = f"{folder_path.name}.zip"

# Creaing a zip archive of the folder
with zipfile.ZipFile(zip_file_name, 'w') as zipf:
    for file_path in folder_path.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, arcname=file_path.relative_to(folder_path))

# Downloading the zip file
files.download(zip_file_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>